In [ ]:
import sys
print(sys.executable)

# Комплексное развитие территорий Москвы и деловые локации

**Цель:** показать территориальное распределение проектов КРТ, их статус,
жилой и нежилой ТЭП, функциональное назначение, распределение показателей
по районам и связь проектов с деловыми локациями.

**Исследовательские вопросы:**
1. В каких районах сосредоточен жилой и нежилой ТЭП КРТ?
2. Как соотносятся проекты в реализации и планируемые?
3. Какая доля проектов и ТЭП приходится на деловые локации?
4. Какие деловые локации получают наибольший прирост нежилых площадей?

**Данные:**
- `OKTMO_districts.geojson` — границы районов (EPSG:32637), население;
- `KRT_MOS.geojson` — проекты КРТ (OGC:CRS84);
- `commerc_districts.geojson` — деловые локации (OGC:CRS84).

In [ ]:
# при необходимости
!pip install geopandas folium branca shapely pyproj  

import re
import json
import html
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely.geometry import Point, Polygon, MultiPolygon, GeometryCollection

import folium
import branca
from folium.plugins import (
    MarkerCluster, HeatMap, Fullscreen, MiniMap,
    MousePosition, Search, MeasureControl
)

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)

for lib in (pd, np, gpd, shapely, folium, branca):
    print(f"{lib.__name__:10s} {lib.__version__}")

In [ ]:
# =====================================================================
# РАЗДЕЛ 3. ПУТИ К ФАЙЛАМ, ПАРАМЕТРЫ И ПОДЛОЖКА
# =====================================================================
import urllib.request
import urllib.error
from pathlib import Path

import folium

# ---------------------------------------------------------------------
# 3.1. Пути к файлам
# ---------------------------------------------------------------------
DATA_RAW = Path("data/raw")
DATA_PROC = Path("data/processed")
DOCS = Path("docs")

DISTRICTS_PATH = DATA_RAW / "OKTMO_districts.geojson"   
KRT_PATH = DATA_RAW / "KRT_MOS.geojson"
BUSINESS_PATH = DATA_RAW / "commerc_districts.geojson"
OUTPUT_PATH = DOCS / "index.html"

for p in (DATA_PROC, DOCS):
    p.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# 3.2. Параметры анализа
# ---------------------------------------------------------------------
METRIC_CRS = 32637   # UTM 37N: центроиды, площади, пространственные соединения
GEO_CRS = 4326       # WGS 84: для Folium

BASE_URL = ""        # домен сайта для относительных ссылок project_url / photo_url

STATUS_IN_PROGRESS = "В реализации"
STATUS_PLANNED = "Планируемый"

# ---------------------------------------------------------------------
BASEMAP_NAME = "CARTO Light"
BASEMAP_URL = (
    "https://basemaps.cartocdn.com/rastertiles/light_all/{z}/{x}/{y}.png"
    "?key=cb1_47fe_1_6fa33fb3987040ac55fd7023"
)
BASEMAP_ATTR = "&copy; OpenStreetMap contributors &copy; CARTO"
BASEMAP_MAX_ZOOM = 20


def build_basemap(show=True):
    """Создаёт слой подложки CARTO."""
    return folium.TileLayer(
        tiles=BASEMAP_URL,
        attr=BASEMAP_ATTR,
        name=BASEMAP_NAME,
        max_zoom=BASEMAP_MAX_ZOOM,
        overlay=False,
        control=True,
        show=show,
    )


def check_basemap(timeout=8):
    """Скачивает тестовый тайл (центр Москвы, z=10) и сообщает результат."""
    test_url = BASEMAP_URL.replace("{z}", "10").replace("{x}", "619").replace("{y}", "320")
    req = urllib.request.Request(test_url, headers={"User-Agent": "moscow-krt-map/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            ctype = resp.headers.get("Content-Type", "")
            if resp.status == 200 and "image" in ctype:
                print(f"✅ Подложка «{BASEMAP_NAME}» работает")
            else:
                print(f"Ответ {resp.status}, тип {ctype}")
    except urllib.error.HTTPError as e:
        print({401: "401 — неверный ключ",
               403: "403 — ключ отклонён или запрещён для этого домена",
               404: "404 — ошибка в адресе"}.get(e.code, f"HTTP {e.code}"))
    except Exception as e:
        print(f" Нет соединения ({type(e).__name__})")


check_basemap()

## 4. Загрузка данных
Читаем слой и выводим CRS, число объектов, типы геометрии и список полей.

In [ ]:
districts_raw = gpd.read_file(DISTRICTS_PATH)
krt_raw = gpd.read_file(KRT_PATH)
business_raw = gpd.read_file(BUSINESS_PATH)

def describe_layer(name, gdf):
    print(f"=== {name} ===")
    print("CRS:", gdf.crs)
    print("Объектов:", len(gdf))
    print("Типы геометрии:", gdf.geom_type.value_counts(dropna=False).to_dict())
    print("Поля:", list(gdf.columns), "\n")

describe_layer("Районы", districts_raw)
describe_layer("КРТ", krt_raw)
describe_layer("Деловые локации", business_raw)

# Если у какого-то слоя CRS не прочитался — задаём явно
if districts_raw.crs is None:
    districts_raw = districts_raw.set_crs(METRIC_CRS)
if krt_raw.crs is None:
    krt_raw = krt_raw.set_crs(GEO_CRS)
if business_raw.crs is None:
    business_raw = business_raw.set_crs(GEO_CRS)

## 5. Очистка данных
### 5.1. Геометрии
Удаляем пустые геометрии, исправляем невалидные (`make_valid`, запасной
вариант — `buffer(0)`). Если после исправления образуется GeometryCollection,
из неё сохраняются только полигоны.

In [ ]:
def keep_polygons(geom):
    """Оставляет только полигональную часть геометрии."""
    if geom is None or geom.is_empty:
        return None
    if isinstance(geom, (Polygon, MultiPolygon)):
        return geom
    if isinstance(geom, GeometryCollection):
        polys = []
        for g in geom.geoms:
            if isinstance(g, Polygon):
                polys.append(g)
            elif isinstance(g, MultiPolygon):
                polys.extend(g.geoms)
        return MultiPolygon(polys) if polys else None
    return None


def fix_geometries(gdf, layer_name):
    gdf = gdf.copy()
    empty = gdf.geometry.isna() | gdf.geometry.is_empty
    invalid = (~empty) & (~gdf.geometry.is_valid)

    report = {
        "Слой": layer_name,
        "Всего объектов": len(gdf),
        "Пустых геометрий": int(empty.sum()),
        "Невалидных геометрий": int(invalid.sum()),
    }
    invalid_table = gdf.loc[invalid].drop(columns="geometry").copy()

    if invalid.any():
        try:
            gdf.loc[invalid, "geometry"] = gdf.loc[invalid].geometry.make_valid()
            report["Метод исправления"] = "make_valid"
        except AttributeError:
            gdf.loc[invalid, "geometry"] = gdf.loc[invalid].geometry.buffer(0)
            report["Метод исправления"] = "buffer(0)"
    else:
        report["Метод исправления"] = "не требовалось"

    gdf = gdf.loc[~empty].copy()
    gdf["geometry"] = gdf.geometry.apply(keep_polygons)
    gdf = gdf[gdf.geometry.notna()].copy()
    report["Осталось объектов"] = len(gdf)
    return gdf.reset_index(drop=True), report, invalid_table


districts_clean, rep_d, invalid_districts = fix_geometries(districts_raw, "Районы")
krt_clean, rep_k, invalid_krt = fix_geometries(krt_raw, "КРТ")
business_clean, rep_b, invalid_business = fix_geometries(business_raw, "Деловые локации")

geometry_report = pd.DataFrame([rep_d, rep_k, rep_b])
geometry_report

In [ ]:
def find_column(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Не найдено ни одно из полей: {candidates}")
    return None

def norm_name(s):
    """Нормализация названия района для сравнения."""
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s).lower().replace("ё", "е")
    s = re.sub(r"[\s\-–—]+", " ", s).strip()
    return s

name_col = find_column(districts_clean, ["NAME", "name", "Name", "district", "rayon", "NAME_RU"])
pop_col = find_column(districts_clean, ["Popul", "population", "POP", "pop"], required=False)

districts = districts_clean[[name_col] + ([pop_col] if pop_col else []) + ["geometry"]].rename(
    columns={name_col: "district_name", pop_col: "population"} if pop_col else {name_col: "district_name"}
)
if "population" not in districts.columns:
    districts["population"] = np.nan

districts["population"] = pd.to_numeric(districts["population"], errors="coerce")
districts["district_id"] = [f"d{i}" for i in range(len(districts))]
districts["district_norm"] = districts["district_name"].apply(norm_name)

districts_m = districts.to_crs(METRIC_CRS)
districts = districts.to_crs(GEO_CRS)

print("Районов:", len(districts))
print("Районов без населения:", districts["population"].isna().sum())
districts.drop(columns="geometry").head()

### 5.3. Поля КРТ и числовые показатели
Переименовываем поля. ТЭП, рабочие места и площадь записаны строками,
поэтому переводим их в числа. Значения, которые не удалось распознать,
выводим в отдельную таблицу. Пустые компоненты ТЭП принимаются равными 0.
Пустые рабочие места помечаются флагом `jobs_missing`.
Поле `Продоно?` сохраняется как `extra_unclear`, но в расчётах не участвует.

In [ ]:
KRT_RENAME = {
    "name": "project_name",
    "status": "status",
    "okrug": "okrug",
    "rayon": "source_district",
    "Рабочие места": "jobs",
    "Общая площадь": "tep_total",
    "Нежилое назначение": "tep_nonresidential",
    "Общественно деловая застройка": "tep_public_business",
    "Жилое назначение": "tep_housing",
    "Продоно?": "extra_unclear",
    "Плоащадь КРТ": "site_area_ha",
    "photo_url": "photo_url",
    "project_url": "project_url",
}

missing_fields = [c for c in KRT_RENAME if c not in krt_clean.columns]
print("Отсутствующие поля КРТ:", missing_fields or "нет")

krt = krt_clean.rename(columns=KRT_RENAME)
for c in KRT_RENAME.values():
    if c not in krt.columns:
        krt[c] = np.nan

krt = krt[list(KRT_RENAME.values()) + ["geometry"]].copy()
krt = krt.to_crs(GEO_CRS)

NUMERIC_COLS = ["jobs", "tep_total", "tep_nonresidential", "tep_public_business",
                "tep_housing", "extra_unclear", "site_area_ha"]
TEP_COMPONENTS = ["tep_housing", "tep_nonresidential", "tep_public_business"]

def to_number(series):
    s = (series.astype(str)
               .str.replace("\u00a0", "", regex=False)
               .str.replace(" ", "", regex=False)
               .str.replace(",", ".", regex=False)
               .str.strip())
    s = s.replace({"": np.nan, "None": np.nan, "nan": np.nan, "null": np.nan, "NaN": np.nan})
    return pd.to_numeric(s, errors="coerce")

bad_values = []
for c in NUMERIC_COLS:
    raw = krt[c]
    parsed = to_number(raw)
    raw_str = raw.astype(str).str.strip()
    is_empty = raw.isna() | raw_str.isin(["", "None", "nan", "null"])
    bad = parsed.isna() & ~is_empty
    for idx in krt.index[bad]:
        bad_values.append({"project_name": krt.at[idx, "project_name"], "поле": c, "значение": raw.at[idx]})
    krt[c + "_was_empty"] = is_empty
    krt[c] = parsed

nonnumeric_table = pd.DataFrame(bad_values, columns=["project_name", "поле", "значение"])

empty_table = pd.DataFrame({
    c: [int(krt[c + "_was_empty"].sum())] for c in NUMERIC_COLS
}, index=["пустых значений"]).T

krt["jobs_missing"] = krt["jobs"].isna()
for c in ["tep_total"] + TEP_COMPONENTS:
    krt[c] = krt[c].fillna(0)
krt["jobs"] = krt["jobs"].fillna(0)

krt = krt.drop(columns=[c + "_was_empty" for c in NUMERIC_COLS])

print("Нечисловые значения:", len(nonnumeric_table))
display(empty_table)
nonnumeric_table

In [ ]:
krt["status_raw"] = krt["status"]

status_key = (krt["status"].fillna("").astype(str).str.strip()
              .str.lower().str.replace("ё", "е"))

STATUS_MAP = {
    "в реализации": STATUS_IN_PROGRESS,
    "реализуется": STATUS_IN_PROGRESS,
    "планируемый": STATUS_PLANNED,
    "планируемые": STATUS_PLANNED,
    "планируется": STATUS_PLANNED,
}

krt["status"] = status_key.map(STATUS_MAP)
krt.loc[krt["status"].isna() & (status_key != ""), "status"] = krt["status_raw"].astype(str).str.strip()
krt["status"] = krt["status"].fillna("Статус не указан")

status_table = (krt.groupby(["status_raw", "status"], dropna=False)
                   .size().rename("проектов").reset_index())
status_table

### 5.5. Дубликаты
**Подтверждённый дубликат** — записи с одинаковыми названием, нормализованной
геометрией и всеми компонентами ТЭП. Такие записи удаляются, остаётся первая.
Все варианты `project_url` сохраняются в поле `project_urls_all`.

**Подозрительные совпадения** (одинаковое название, но другая геометрия или
ТЭП) не удаляются автоматически, а выводятся для ручной проверки.

In [ ]:
def geom_key(g):
    try:
        return g.normalize().wkb_hex
    except Exception:
        return g.wkt

krt["_geom_key"] = krt.geometry.apply(geom_key)
DUP_KEY = ["project_name", "_geom_key", "tep_total"] + TEP_COMPONENTS

dup_mask = krt.duplicated(DUP_KEY, keep=False)
duplicates_table = (krt.loc[dup_mask, ["project_name", "status", "source_district",
                                       "tep_total", "tep_housing", "project_url"]]
                       .sort_values("project_name"))

krt["project_urls_all"] = krt.groupby(DUP_KEY)["project_url"].transform(
    lambda x: "; ".join(sorted(set(x.dropna().astype(str))))
)

n_before = len(krt)
krt = krt.drop_duplicates(DUP_KEY, keep="first").reset_index(drop=True)
print(f"Удалено подтверждённых дубликатов: {n_before - len(krt)}")

name_dup = krt[krt.duplicated("project_name", keep=False)]
suspected_duplicates_table = name_dup[["project_name", "status", "source_district",
                                       "tep_total", "project_url"]].sort_values("project_name")
print("Подозрительных совпадений по названию (не удалены):", len(suspected_duplicates_table))

krt["krt_id"] = np.arange(len(krt))
duplicates_table

In [ ]:
tech_cols = [c for c in business_clean.columns
             if c.startswith("auxiliary_storage_labeling")
             or c in ("path", "layer", "iconCaption", "marker-color", "id")]

business = business_clean.drop(columns=tech_cols).copy()

business = business.rename(columns={
    "Name": "business_name",
    "description": "business_description",
    "main": "business_main",
    "fill": "style_fill",
    "fill-opacity": "style_fill_opacity",
    "stroke": "style_stroke",
    "stroke-width": "style_stroke_width",
    "stroke-opacity": "style_stroke_opacity",
})
for c in ["fid", "business_name", "business_description", "business_main", "style_fill",
          "style_fill_opacity", "style_stroke", "style_stroke_width", "style_stroke_opacity"]:
    if c not in business.columns:
        business[c] = np.nan

business["business_name"] = (business["business_name"]
                             .fillna(business["business_description"])
                             .fillna("Локация " + business["fid"].astype(str)))
business["business_description"] = business["business_description"].fillna("—")
business["business_main"] = business["business_main"].astype("string").fillna("не указан")

for c in ["style_fill_opacity", "style_stroke_width", "style_stroke_opacity"]:
    business[c] = pd.to_numeric(business[c], errors="coerce")

business["business_id"] = [f"b{i}" for i in range(len(business))]
business = business.to_crs(GEO_CRS)
business_m = business.to_crs(METRIC_CRS)

print("Удалено технических полей:", len(tech_cols))
print("Коды main:", business["business_main"].value_counts().to_dict())
business.drop(columns="geometry")

## 6. Подготовка ТЭП
- `tep_nonres` = нежилое назначение + общественно-деловая застройка;
- `tep_components_sum` = жилое + нежилое + общественно-деловое;
- проверка: `tep_total` должен совпадать с суммой компонентов (допуск 1 м²).

In [ ]:
krt["tep_nonres"] = krt["tep_nonresidential"] + krt["tep_public_business"]
krt["tep_components_sum"] = krt["tep_housing"] + krt["tep_nonresidential"] + krt["tep_public_business"]
krt["tep_diff"] = krt["tep_total"] - krt["tep_components_sum"]
krt["tep_check_ok"] = krt["tep_diff"].abs() <= 1

tep_mismatch_table = krt.loc[~krt["tep_check_ok"],
    ["project_name", "tep_total", "tep_housing", "tep_nonresidential",
     "tep_public_business", "tep_components_sum", "tep_diff"]]

krt_m = krt.to_crs(METRIC_CRS)
krt["area_ha_geom"] = (krt_m.geometry.area / 10_000).round(2)
krt["area_ratio"] = np.where(krt["site_area_ha"] > 0, krt["area_ha_geom"] / krt["site_area_ha"], np.nan)
area_check_table = krt.loc[(krt["area_ratio"] < 0.7) | (krt["area_ratio"] > 1.3),
    ["project_name", "site_area_ha", "area_ha_geom", "area_ratio"]]

print(f"Совпадает с суммой компонентов: {krt['tep_check_ok'].sum()} из {len(krt)}")
print(f"Площадь отличается от геометрии более чем на 30%: {len(area_check_table)}")
tep_mismatch_table

## классификация крт

In [ ]:
conditions = [
    krt["tep_housing"] > 0,
    (krt["tep_housing"] == 0) & ((krt["tep_nonresidential"] > 0) | (krt["tep_public_business"] > 0)),
]
choices = ["Жилое КРТ", "Нежилое КРТ"]
krt["functional_type"] = np.select(conditions, choices, default="Назначение не определено")

# Индикаторы для агрегации
krt["is_in_progress"] = (krt["status"] == STATUS_IN_PROGRESS).astype(int)
krt["is_planned"] = (krt["status"] == STATUS_PLANNED).astype(int)
krt["is_other_status"] = 1 - krt["is_in_progress"] - krt["is_planned"]
krt["is_housing"] = (krt["functional_type"] == "Жилое КРТ").astype(int)
krt["is_nonres"] = (krt["functional_type"] == "Нежилое КРТ").astype(int)
krt["is_undefined"] = (krt["functional_type"] == "Назначение не определено").astype(int)

zero_tep_table = krt.loc[krt["is_undefined"] == 1, ["project_name", "status", "source_district", "tep_total"]]

pd.crosstab(krt["functional_type"], krt["status"], margins=True, margins_name="Итого")

## расчет Центроидов

In [ ]:
krt_m = krt.to_crs(METRIC_CRS)
cent_m = gpd.GeoDataFrame(krt_m[["krt_id"]].copy(), geometry=krt_m.geometry.centroid, crs=METRIC_CRS)

krt["centroid_inside_polygon"] = cent_m.geometry.within(krt_m.geometry).values

cent_geo = cent_m.to_crs(GEO_CRS)
krt["lon"] = cent_geo.geometry.x.values
krt["lat"] = cent_geo.geometry.y.values

centroid_outside_table = krt.loc[~krt["centroid_inside_polygon"],
    ["project_name", "status", "source_district", "lat", "lon"]]
krt["geom_type"] = krt.geom_type

print("Центроидов вне собственного полигона:", len(centroid_outside_table))
centroid_outside_table

## привязываем к районам

In [ ]:
j_d = gpd.sjoin(
    cent_m,
    districts_m[["district_id", "district_name", "district_norm", "geometry"]],
    how="left", predicate="within"
)

multi_district_table = j_d[j_d.duplicated("krt_id", keep=False)].merge(
    krt[["krt_id", "project_name"]], on="krt_id")[["krt_id", "project_name", "district_name"]]
j_d = j_d.drop_duplicates("krt_id")

krt = krt.merge(
    j_d[["krt_id", "district_id", "district_name", "district_norm"]].rename(columns={
        "district_id": "centroid_district_id",
        "district_name": "centroid_district",
        "district_norm": "centroid_district_norm",
    }),
    on="krt_id", how="left"
)

krt["source_district_norm"] = krt["source_district"].apply(norm_name)
krt["district_match"] = krt["source_district_norm"] == krt["centroid_district_norm"].fillna("")

district_not_found_table = krt.loc[krt["centroid_district"].isna(),
    ["project_name", "source_district", "okrug", "lat", "lon"]]
district_mismatch_table = krt.loc[krt["centroid_district"].notna() & ~krt["district_match"],
    ["project_name", "okrug", "source_district", "centroid_district", "centroid_inside_polygon"]]

krt["centroid_district"] = krt["centroid_district"].fillna("Район не определён")

print("Район по центроиду не найден:", len(district_not_found_table))
print("Несовпадение исходного района и района по центроиду:", len(district_mismatch_table))
district_mismatch_table

## агрегация по районам

In [ ]:
SUM_COLS = ["is_in_progress", "is_planned", "is_other_status",
            "is_housing", "is_nonres", "is_undefined",
            "tep_total", "tep_housing", "tep_nonres",
            "tep_nonresidential", "tep_public_business", "jobs"]

AGG_RENAME = {
    "is_in_progress": "krt_in_progress", "is_planned": "krt_planned",
    "is_other_status": "krt_other_status", "is_housing": "krt_housing",
    "is_nonres": "krt_nonres", "is_undefined": "krt_undefined",
}

def aggregate_krt(df, by):
    out = df.groupby(by)[SUM_COLS].sum()
    out.insert(0, "krt_count", df.groupby(by).size())
    return out.rename(columns=AGG_RENAME).reset_index()

AGG_COLS = ["krt_count"] + [AGG_RENAME.get(c, c) for c in SUM_COLS]

krt_df = pd.DataFrame(krt.drop(columns="geometry"))

d_agg = aggregate_krt(krt_df.dropna(subset=["centroid_district_id"]), "centroid_district_id")
d_agg = d_agg.rename(columns={"centroid_district_id": "district_id"})

districts_agg = districts.merge(d_agg, on="district_id", how="left")
districts_agg[AGG_COLS] = districts_agg[AGG_COLS].fillna(0)

pop = districts_agg["population"].where(districts_agg["population"] > 0)
districts_agg["tep_housing_per_capita"] = (districts_agg["tep_housing"] / pop).round(3)
districts_agg["tep_nonres_per_capita"] = (districts_agg["tep_nonres"] / pop).round(3)

districts_agg.drop(columns="geometry").sort_values("tep_housing", ascending=False).head(15)

## Связываем с деловыми локациями

In [ ]:
j_b = gpd.sjoin(
    cent_m,
    business_m[["business_id", "business_name", "business_main", "geometry"]],
    how="inner", predicate="within"
)[["krt_id", "business_id", "business_name", "business_main"]]

bus_per_krt = j_b.groupby("krt_id").agg(
    business_location=("business_name", lambda x: "; ".join(sorted(set(x)))),
    business_main_codes=("business_main", lambda x: "; ".join(sorted(set(map(str, x))))),
    business_count=("business_id", "nunique"),
).reset_index()

krt = krt.merge(bus_per_krt, on="krt_id", how="left")
krt["business_count"] = krt["business_count"].fillna(0).astype(int)
krt["inside_business_location"] = np.where(krt["business_count"] > 0, "Да", "Нет")
krt["business_location"] = krt["business_location"].fillna("Вне деловых локаций")
krt["business_main_codes"] = krt["business_main_codes"].fillna("—")

multi_business_table = krt.loc[krt["business_count"] > 1,
    ["project_name", "business_location", "business_main_codes", "business_count"]]

# Агрегация: проект учитывается в каждой локации, куда попал его центроид
krt_df = pd.DataFrame(krt.drop(columns="geometry"))
b_long = j_b[["krt_id", "business_id"]].merge(krt_df[["krt_id"] + SUM_COLS], on="krt_id")
b_agg = aggregate_krt(b_long, "business_id")

business_agg = business.merge(b_agg, on="business_id", how="left")
business_agg[AGG_COLS] = business_agg[AGG_COLS].fillna(0)

print("КРТ в деловых локациях:", (krt["inside_business_location"] == "Да").sum(), "из", len(krt))
print("КРТ сразу в нескольких локациях:", len(multi_business_table))
business_agg.drop(columns="geometry")[["business_name", "business_main"] + AGG_COLS] \
    .sort_values("tep_total", ascending=False)

In [ ]:
control_tables = {
    "Невалидные геометрии КРТ": invalid_krt,
    "Нечисловые значения": nonnumeric_table,
    "Подтверждённые дубликаты (удалены)": duplicates_table,
    "Совпадение названий (на проверку)": suspected_duplicates_table,
    "Несовпадение ТЭП с суммой компонентов": tep_mismatch_table,
    "Площадь КРТ ≠ площади полигона (>30%)": area_check_table,
    "Нулевые значения всех компонентов ТЭП": zero_tep_table,
    "Центроид вне собственного полигона": centroid_outside_table,
    "Район по центроиду не найден": district_not_found_table,
    "Центроид в нескольких районах": multi_district_table,
    "Несовпадение исходного района и района по центроиду": district_mismatch_table,
    "КРТ в нескольких деловых локациях": multi_business_table,
}

summary = pd.DataFrame({"Записей": {k: len(v) for k, v in control_tables.items()}})
display(geometry_report)
display(status_table)
display(summary)

for title, table in control_tables.items():
    if len(table):
        print(f"\n### {title}")
        display(table.head(30))

In [ ]:
# Сохранение обработанных слоёв
krt.drop(columns=["_geom_key"]).to_file(DATA_PROC / "KRT_MOS_processed.geojson", driver="GeoJSON")
districts_agg.drop(columns=["district_norm"]).to_file(DATA_PROC / "OKTMO_districts_processed.geojson", driver="GeoJSON")
business_agg.to_file(DATA_PROC / "commerc_districts_processed.geojson", driver="GeoJSON")
print("Обработанные данные сохранены в", DATA_PROC)

Создание карты folium

In [ ]:
def fmt_num(x, digits=0):
    if x is None or pd.isna(x):
        return "—"
    return f"{x:,.{digits}f}".replace(",", " ")

def add_fmt(df, cols, digits=0):
    for c in cols:
        df[c + "_fmt"] = df[c].apply(lambda v: fmt_num(v, digits))
    return df

def gdf_to_geojson(gdf, cols):
    """GeoDataFrame -> dict GeoJSON только с нужными полями (безопасно для JSON)."""
    return json.loads(gdf[cols + ["geometry"]].to_crs(GEO_CRS).to_json())

def make_abs_url(u):
    if u is None or pd.isna(u) or str(u).strip() == "":
        return None
    u = str(u).strip()
    if u.startswith(("http://", "https://")):
        return u
    return BASE_URL.rstrip("/") + u if BASE_URL else None

def make_bins(series):
    """Классы для картограммы: отдельный класс «0» + квантили положительных значений."""
    s = series.fillna(0)
    pos = s[s > 0]
    vmax = float(s.max())
    if len(pos) < 5 or vmax <= 1:
        return list(np.linspace(0, max(vmax, 1), 6))
    qs = np.quantile(pos, [0.2, 0.4, 0.6, 0.8])
    bins = sorted(set([0.0, 1.0] + [float(q) for q in qs if q > 1] + [vmax]))
    if len(bins) < 4:
        bins = list(np.linspace(0, vmax, 6))
    return bins

FMT_COLS = ["tep_total", "tep_housing", "tep_nonres", "tep_public_business", "jobs"]

# --- Районы
districts_map = districts_agg.copy()
add_fmt(districts_map, FMT_COLS + ["population"])
add_fmt(districts_map, ["tep_housing_per_capita", "tep_nonres_per_capita"], digits=2)
for c in ["krt_count", "krt_housing", "krt_nonres", "krt_in_progress", "krt_planned"]:
    districts_map[c] = districts_map[c].astype(int)

DISTRICT_TT_FIELDS = ["district_name", "population_fmt", "krt_count", "krt_in_progress", "krt_planned",
                      "krt_housing", "krt_nonres", "tep_housing_fmt", "tep_nonres_fmt",
                      "tep_total_fmt", "jobs_fmt", "tep_housing_per_capita_fmt", "tep_nonres_per_capita_fmt"]
DISTRICT_TT_ALIASES = ["Район", "Население", "КРТ всего", "В реализации", "Планируемые",
                       "Жилых КРТ", "Нежилых КРТ", "Жилой ТЭП, м²", "Нежилой ТЭП, м²",
                       "Общий ТЭП, м²", "Рабочие места", "Жилой ТЭП на жителя, м²",
                       "Нежилой ТЭП на жителя, м²"]

# --- Деловые локации
business_map = business_agg.copy()
add_fmt(business_map, FMT_COLS)
for c in ["krt_count", "krt_housing", "krt_nonres", "krt_in_progress", "krt_planned"]:
    business_map[c] = business_map[c].astype(int)

# --- КРТ
krt_map = krt.drop(columns=["_geom_key"]).copy()
add_fmt(krt_map, FMT_COLS + ["tep_nonresidential"])
add_fmt(krt_map, ["site_area_ha"], digits=2)
krt_map["jobs_fmt"] = np.where(krt_map["jobs_missing"], "нет данных", krt_map["jobs_fmt"])

def build_popup(r):
    e = lambda v: html.escape("—" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v))
    url = make_abs_url(r["project_url"])
    photo = make_abs_url(r["photo_url"])
    if photo and "default-map-project" in photo:
        photo = None
    rows = [
        ("Статус", r["status"]), ("Назначение", r["functional_type"]),
        ("Округ", r["okrug"]), ("Район (исходный)", r["source_district"]),
        ("Район по центроиду", r["centroid_district"]),
        ("Общий ТЭП, м²", r["tep_total_fmt"]), ("Жилой ТЭП, м²", r["tep_housing_fmt"]),
        ("Нежилой ТЭП (всего), м²", r["tep_nonres_fmt"]),
        ("— нежилое назначение, м²", r["tep_nonresidential_fmt"]),
        ("— общественно-деловой, м²", r["tep_public_business_fmt"]),
        ("Площадь территории, га", r["site_area_ha_fmt"]),
        ("Рабочие места", r["jobs_fmt"]),
        ("В деловой локации", r["inside_business_location"]),
        ("Деловая локация", r["business_location"]),
    ]
    table = "".join(f"<tr><td style='color:#555;padding-right:8px'>{e(k)}</td><td><b>{e(v)}</b></td></tr>"
                    for k, v in rows)
    parts = [f"<div style='font-size:12px;max-width:360px'><h4 style='margin:0 0 6px'>{e(r['project_name'])}</h4>"]
    if photo:
        parts.append(f"<img src='{html.escape(photo)}' style='width:100%;max-height:180px;object-fit:cover;margin-bottom:6px'>")
    parts.append(f"<table>{table}</table>")
    if url:
        parts.append(f"<p style='margin-top:6px'><a href='{html.escape(url)}' target='_blank'>Страница проекта →</a></p>")
    parts.append("</div>")
    return "".join(parts)

krt_map["popup_html"] = krt_map.apply(build_popup, axis=1)

KRT_TT_FIELDS = ["project_name", "status", "functional_type", "centroid_district", "tep_total_fmt", "business_location"]
KRT_TT_ALIASES = ["Проект", "Статус", "Назначение", "Район", "Общий ТЭП, м²", "Деловая локация"]

создание слоев

In [ ]:
# =====================================================================
# 12.2. Оформление, панели и функции слоёв
# =====================================================================
DISTRICT_FILL_OPACITY = 0.2
DISTRICT_LINE_OPACITY = 0.2
DISTRICT_LINE_WEIGHT = 0.2

KRT_FILL_OPACITY = 0.78
KRT_LINE_OPACITY = 1.00
KRT_LINE_WEIGHT = 2

BUSINESS_MAX_FILL_OPACITY = 0.15  # верхний предел заливки деловых локаций

# Чем больше z-index, тем выше слой
PANES = {
    "districts_pane": 350,
    "business_pane": 380,
    "diagrams_pane": 420,
    "krt_pane": 450,
    "centroids_pane": 460,
}


def ensure_panes(m):
    """Создаёт панели на карте один раз. Безопасно вызывать многократно."""
    if getattr(m, "_krt_panes_ready", False):
        return
    for pane_name, z in PANES.items():
        folium.map.CustomPane(pane_name, z_index=z, pointer_events=True).add_to(m)
    m._krt_panes_ready = True


add_panes = ensure_panes  # совместимость со старым кодом 12.3


def set_pane(layer, pane_name):
    """Помещает GeoJson-слой в заданную панель."""
    if getattr(layer, "options", None) is None:
        layer.options = {}
    layer.options["pane"] = pane_name
    return layer


def add_choropleth(m, gdf, value_col, name, cmap, legend, show):
    ensure_panes(m)
    ch = folium.Choropleth(
        geo_data=gdf_to_geojson(gdf, ["district_id"] + DISTRICT_TT_FIELDS),
        data=gdf, columns=["district_id", value_col],
        key_on="feature.properties.district_id",
        bins=make_bins(gdf[value_col]),
        fill_color=cmap,
        fill_opacity=DISTRICT_FILL_OPACITY,
        line_opacity=DISTRICT_LINE_OPACITY,
        line_weight=DISTRICT_LINE_WEIGHT,
        nan_fill_color="white", legend_name=legend,
        name=name, overlay=True, show=show, highlight=True,
    )
    set_pane(ch.geojson, "districts_pane")
    ch.geojson.add_child(folium.GeoJsonTooltip(
        fields=DISTRICT_TT_FIELDS, aliases=DISTRICT_TT_ALIASES, sticky=True))
    ch.add_to(m)
    return ch


def add_krt_polygons(m, gdf, name, color, show,
                     fill_opacity=KRT_FILL_OPACITY, weight=KRT_LINE_WEIGHT):
    ensure_panes(m)
    if gdf.empty:
        print(f"Слой «{name}» пуст — пропущен")
        return None
    layer = folium.GeoJson(
        gdf_to_geojson(gdf, KRT_TT_FIELDS + ["popup_html"]),
        name=name, show=show,
        style_function=lambda f, c=color, fo=fill_opacity, w=weight: {
            "color": c, "weight": w, "opacity": KRT_LINE_OPACITY,
            "fillColor": c, "fillOpacity": fo},
        highlight_function=lambda f, fo=fill_opacity, w=weight: {
            "weight": w + 2, "fillOpacity": min(fo + 0.15, 0.9)},
        tooltip=folium.GeoJsonTooltip(fields=KRT_TT_FIELDS, aliases=KRT_TT_ALIASES, sticky=True),
        popup=folium.GeoJsonPopup(fields=["popup_html"], labels=False, max_width=380),
    )
    set_pane(layer, "krt_pane")
    layer.add_to(m)
    return layer


def add_proportional_circles(m, df, value_col, name, color, label, min_radius=3, max_radius=28):
    ensure_panes(m)
    fg = folium.FeatureGroup(name=name, show=False)
    d = df[df[value_col] > 0].sort_values(value_col, ascending=False)  # мелкие поверх крупных
    if not d.empty:
        scale = (max_radius - min_radius) / np.sqrt(d[value_col].max())
        for _, r in d.iterrows():
            radius = min_radius + scale * np.sqrt(r[value_col])
            folium.CircleMarker(
                [r["lat"], r["lon"]], radius=float(radius),
                color=color, weight=1, fill=True, fill_color=color, fill_opacity=0.45,
                pane="diagrams_pane",
                tooltip=f"<b>{html.escape(str(r['project_name']))}</b><br>{label}: {fmt_num(r[value_col])} м²",
            ).add_to(fg)
    fg.add_to(m)
    return fg


def add_heatmap(m, df, value_col, name):
    fg = folium.FeatureGroup(name=name, show=False)
    d = df[df[value_col] > 0]
    if not d.empty:
        w = np.log1p(d[value_col])
        w = w / w.max()
        points = [[float(a), float(b), float(c)] for a, b, c in zip(d["lat"], d["lon"], w)]
        HeatMap(points, radius=25, blur=18, min_opacity=0.25, max_zoom=13).add_to(fg)
    fg.add_to(m)
    return fg


def business_style(feature):
    p = feature["properties"]
    val = lambda k, default: default if p.get(k) is None else p.get(k)
    return {
        "fillColor": val("style_fill", "#ff7f00"),
        "fillOpacity": min(val("style_fill_opacity", 0.15), BUSINESS_MAX_FILL_OPACITY),
        "color": val("style_stroke", "#ff7f00"),
        "weight": val("style_stroke_width", 2),
        "opacity": val("style_stroke_opacity", 0.9),
        "dashArray": "6 4",
    }

In [ ]:
# =====================================================================
# 12.3. Сборка карты
# =====================================================================
center = [krt_map["lat"].mean(), krt_map["lon"].mean()]
m = folium.Map(location=center, zoom_start=10, tiles=None, control_scale=True)

# --- Подложка и панели
build_basemap(show=True).add_to(m)
add_panes(m)

# --- Картограммы районов (districts_pane, нижний уровень)
ch_housing = add_choropleth(m, districts_map, "tep_housing", "Районы — жилой ТЭП КРТ",
                            "YlOrRd", "Суммарный жилой ТЭП КРТ, м²", show=True)
ch_nonres = add_choropleth(m, districts_map, "tep_nonres", "Районы — нежилой ТЭП КРТ",
                           "PuBu", "Суммарный нежилой ТЭП КРТ (нежилое + общ.-деловое), м²", show=False)

# --- Деловые локации (business_pane)
BUS_TT_FIELDS = ["business_name", "business_description", "business_main", "krt_count",
                 "krt_housing", "krt_nonres", "tep_total_fmt", "tep_housing_fmt",
                 "tep_nonres_fmt", "jobs_fmt"]
BUS_TT_ALIASES = ["Деловая локация", "Описание", "Код main", "КРТ всего", "Жилых КРТ",
                  "Нежилых КРТ", "Общий ТЭП, м²", "Жилой ТЭП, м²", "Нежилой ТЭП, м²", "Рабочие места"]

business_layer = folium.GeoJson(
    gdf_to_geojson(business_map, BUS_TT_FIELDS + ["style_fill", "style_fill_opacity", "style_stroke",
                                                  "style_stroke_width", "style_stroke_opacity"]),
    name="Деловые локации", show=True,
    style_function=business_style,
    highlight_function=lambda f: {"weight": 4, "fillOpacity": 0.25},
    tooltip=folium.GeoJsonTooltip(fields=BUS_TT_FIELDS, aliases=BUS_TT_ALIASES, sticky=True),
)
set_pane(business_layer, "business_pane")
business_layer.add_to(m)

# --- Тепловые карты (overlayPane)
add_heatmap(m, krt_map, "tep_housing", "Тепловая карта — жилой ТЭП")
add_heatmap(m, krt_map, "tep_nonres", "Тепловая карта — нежилой ТЭП")

# --- Картодиаграммы (diagrams_pane)
add_proportional_circles(m, krt_map, "tep_housing", "Картодиаграмма — жилой ТЭП", "#31a354", "Жилой ТЭП")
add_proportional_circles(m, krt_map, "tep_nonres", "Картодиаграмма — нежилой ТЭП", "#756bb1", "Нежилой ТЭП")

# --- Полигоны КРТ (krt_pane, выше всех тематических слоёв)
add_krt_polygons(m, krt_map[krt_map["status"] == STATUS_IN_PROGRESS], "КРТ — в реализации", "#e34a33", show=True)
add_krt_polygons(m, krt_map[krt_map["status"] == STATUS_PLANNED], "КРТ — планируемые", "#2c7fb8", show=True)
other = krt_map[~krt_map["status"].isin([STATUS_IN_PROGRESS, STATUS_PLANNED])]
add_krt_polygons(m, other, "КРТ — иной статус", "#636363", show=False)

add_krt_polygons(m, krt_map[krt_map["functional_type"] == "Жилое КРТ"], "КРТ — жилое назначение", "#31a354", show=False)
add_krt_polygons(m, krt_map[krt_map["functional_type"] == "Нежилое КРТ"], "КРТ — нежилое назначение", "#756bb1", show=False)
add_krt_polygons(m, krt_map[krt_map["functional_type"] == "Назначение не определено"],
                 "КРТ — назначение не определено", "#969696", show=False)

# Слой-обводка: заливка почти прозрачная, толстая яркая граница
add_krt_polygons(m, krt_map[krt_map["inside_business_location"] == "Да"], "КРТ в деловых локациях",
                 "#ffd700", show=False, fill_opacity=0.05, weight=4)

# --- Кластеры (markerPane, остаются кликабельными поверх полигонов)
cluster = MarkerCluster(name="Кластеры проектов КРТ", show=False)
STATUS_COLORS = {STATUS_IN_PROGRESS: "red", STATUS_PLANNED: "blue"}
for _, r in krt_map.iterrows():
    folium.Marker(
        [r["lat"], r["lon"]],
        icon=folium.Icon(color=STATUS_COLORS.get(r["status"], "gray"), icon="info-sign"),
        tooltip=html.escape(str(r["project_name"])),
        popup=folium.Popup(r["popup_html"], max_width=380),
    ).add_to(cluster)
cluster.add_to(m)

# --- Технический слой центроидов (centroids_pane, над полигонами КРТ)
cent_gdf = gpd.GeoDataFrame(
    krt_map[["project_name", "status", "centroid_district", "business_location", "centroid_inside_polygon"]].copy(),
    geometry=gpd.points_from_xy(krt_map["lon"], krt_map["lat"]), crs=GEO_CRS)
cent_gdf["centroid_inside_polygon"] = np.where(cent_gdf["centroid_inside_polygon"], "да", "нет")

centroid_layer = folium.GeoJson(
    json.loads(cent_gdf.to_json()),
    name="Технический слой — центроиды КРТ", show=False,
    marker=folium.CircleMarker(radius=3, color="black", weight=1, fill=True,
                               fill_color="yellow", fill_opacity=1, pane="centroids_pane"),
    tooltip=folium.GeoJsonTooltip(
        fields=["project_name", "centroid_district", "business_location", "centroid_inside_polygon"],
        aliases=["Проект", "Район по центроиду", "Деловая локация", "Центроид внутри полигона"]),
)
set_pane(centroid_layer, "centroids_pane")
centroid_layer.add_to(m)

In [ ]:
# =====================================================================
# 12.3–14. СБОРКА КАРТЫ, ИНСТРУМЕНТЫ И СОХРАНЕНИЕ (одна ячейка)
# =====================================================================

# ---------------------------------------------------------------------
# 1. Карта, подложка, панели
# ---------------------------------------------------------------------
center = [krt_map["lat"].mean(), krt_map["lon"].mean()]
m = folium.Map(location=center, zoom_start=10, tiles=None, control_scale=True)
build_basemap(show=True).add_to(m)
ensure_panes(m)

# ---------------------------------------------------------------------
# 2. Картограммы районов
# ---------------------------------------------------------------------
ch_housing = add_choropleth(m, districts_map, "tep_housing", "Районы — жилой ТЭП КРТ",
                            "YlOrRd", "Суммарный жилой ТЭП КРТ, м²", show=True)
ch_nonres = add_choropleth(m, districts_map, "tep_nonres", "Районы — нежилой ТЭП КРТ",
                           "PuBu", "Суммарный нежилой ТЭП КРТ (нежилое + общ.-деловое), м²", show=False)

# ---------------------------------------------------------------------
# 3. Деловые локации
# ---------------------------------------------------------------------
BUS_TT_FIELDS = ["business_name", "business_description", "business_main", "krt_count",
                 "krt_housing", "krt_nonres", "tep_total_fmt", "tep_housing_fmt",
                 "tep_nonres_fmt", "jobs_fmt"]
BUS_TT_ALIASES = ["Деловая локация", "Описание", "Код main", "КРТ всего", "Жилых КРТ",
                  "Нежилых КРТ", "Общий ТЭП, м²", "Жилой ТЭП, м²", "Нежилой ТЭП, м²", "Рабочие места"]

business_layer = folium.GeoJson(
    gdf_to_geojson(business_map, BUS_TT_FIELDS + ["style_fill", "style_fill_opacity", "style_stroke",
                                                  "style_stroke_width", "style_stroke_opacity"]),
    name="Деловые локации", show=True,
    style_function=business_style,
    highlight_function=lambda f: {"weight": 4, "fillOpacity": 0.25},
    tooltip=folium.GeoJsonTooltip(fields=BUS_TT_FIELDS, aliases=BUS_TT_ALIASES, sticky=True),
)
set_pane(business_layer, "business_pane")
business_layer.add_to(m)

# ---------------------------------------------------------------------
# 4. Тепловые карты и картодиаграммы
# ---------------------------------------------------------------------
add_heatmap(m, krt_map, "tep_housing", "Тепловая карта — жилой ТЭП")
add_heatmap(m, krt_map, "tep_nonres", "Тепловая карта — нежилой ТЭП")
add_proportional_circles(m, krt_map, "tep_housing", "Картодиаграмма — жилой ТЭП", "#31a354", "Жилой ТЭП")
add_proportional_circles(m, krt_map, "tep_nonres", "Картодиаграмма — нежилой ТЭП", "#756bb1", "Нежилой ТЭП")

# ---------------------------------------------------------------------
# 5. Полигоны КРТ (верхний уровень)
# ---------------------------------------------------------------------
add_krt_polygons(m, krt_map[krt_map["status"] == STATUS_IN_PROGRESS], "КРТ — в реализации", "#e34a33", show=True)
add_krt_polygons(m, krt_map[krt_map["status"] == STATUS_PLANNED], "КРТ — планируемые", "#2c7fb8", show=True)
other = krt_map[~krt_map["status"].isin([STATUS_IN_PROGRESS, STATUS_PLANNED])]
add_krt_polygons(m, other, "КРТ — иной статус", "#636363", show=False)

add_krt_polygons(m, krt_map[krt_map["functional_type"] == "Жилое КРТ"], "КРТ — жилое назначение", "#31a354", show=False)
add_krt_polygons(m, krt_map[krt_map["functional_type"] == "Нежилое КРТ"], "КРТ — нежилое назначение", "#756bb1", show=False)
add_krt_polygons(m, krt_map[krt_map["functional_type"] == "Назначение не определено"],
                 "КРТ — назначение не определено", "#969696", show=False)
add_krt_polygons(m, krt_map[krt_map["inside_business_location"] == "Да"], "КРТ в деловых локациях",
                 "#ffd700", show=False, fill_opacity=0.05, weight=4)

# ---------------------------------------------------------------------
# 6. Кластеры
# ---------------------------------------------------------------------
cluster = MarkerCluster(name="Кластеры проектов КРТ", show=False)
STATUS_COLORS = {STATUS_IN_PROGRESS: "red", STATUS_PLANNED: "blue"}
for _, r in krt_map.iterrows():
    folium.Marker(
        [r["lat"], r["lon"]],
        icon=folium.Icon(color=STATUS_COLORS.get(r["status"], "gray"), icon="info-sign"),
        tooltip=html.escape(str(r["project_name"])),
        popup=folium.Popup(r["popup_html"], max_width=380),
    ).add_to(cluster)
cluster.add_to(m)

# ---------------------------------------------------------------------
# 7. Технический слой центроидов
# ---------------------------------------------------------------------
cent_gdf = gpd.GeoDataFrame(
    krt_map[["project_name", "status", "centroid_district", "business_location", "centroid_inside_polygon"]].copy(),
    geometry=gpd.points_from_xy(krt_map["lon"], krt_map["lat"]), crs=GEO_CRS)
cent_gdf["centroid_inside_polygon"] = np.where(cent_gdf["centroid_inside_polygon"], "да", "нет")

centroid_layer = folium.GeoJson(
    json.loads(cent_gdf.to_json()),
    name="Технический слой — центроиды КРТ", show=False,
    marker=folium.CircleMarker(radius=3, color="black", weight=1, fill=True,
                               fill_color="yellow", fill_opacity=1, pane="centroids_pane"),
    tooltip=folium.GeoJsonTooltip(
        fields=["project_name", "centroid_district", "business_location", "centroid_inside_polygon"],
        aliases=["Проект", "Район по центроиду", "Деловая локация", "Центроид внутри полигона"]),
)
set_pane(centroid_layer, "centroids_pane")
centroid_layer.add_to(m)

# ---------------------------------------------------------------------
# 8. Инструменты. Каждый добавляется отдельно: ошибка одного не мешает остальным
# ---------------------------------------------------------------------
def try_add(label, fn):
    try:
        fn()
        print(f"1 {label}")
    except Exception as e:
        print(f"0 {label}: {type(e).__name__}: {e}")

try_add("Полноэкранный режим", lambda: Fullscreen(
    position="topleft", title="Полноэкранный режим", title_cancel="Выйти").add_to(m))
try_add("Мини-карта", lambda: MiniMap(
    tile_layer=build_basemap(), toggle_display=True, position="bottomright").add_to(m))
try_add("Координаты курсора", lambda: MousePosition(
    position="bottomleft", separator=" | ", prefix="Координаты:", num_digits=5).add_to(m))
try_add("Измерения", lambda: MeasureControl(
    position="topleft", primary_length_unit="meters",
    secondary_length_unit="kilometers", primary_area_unit="hectares").add_to(m))
try_add("Поиск района", lambda: Search(
    layer=ch_housing.geojson, geom_type="Polygon", search_label="district_name",
    placeholder="Поиск района", collapsed=True, position="topright").add_to(m))
try_add("Поиск проекта", lambda: Search(
    layer=centroid_layer, geom_type="Point", search_label="project_name",
    placeholder="Поиск проекта КРТ", collapsed=True, position="topright").add_to(m))

title_html = """
<div style="position:fixed;top:10px;left:60px;z-index:9999;background:white;padding:6px 12px;
            border-radius:6px;box-shadow:0 1px 4px rgba(0,0,0,.3);font-family:sans-serif">
  <b>Проекты КРТ Москвы и деловые локации</b><br>
  <span style="font-size:11px;color:#555">Привязка к районам и локациям — по центроиду проекта</span>
</div>"""
m.get_root().html.add_child(branca.element.Element(title_html))

minx, miny, maxx, maxy = districts.total_bounds
m.fit_bounds([[miny, minx], [maxy, maxx]])

# LayerControl — строго последним, после всех слоёв
folium.LayerControl(collapsed=False).add_to(m)
print("1 Переключатель слоёв")

# ---------------------------------------------------------------------
# 9. Сохранение и итоговая проверка
# ---------------------------------------------------------------------
m.save(str(OUTPUT_PATH))
page = OUTPUT_PATH.read_text(encoding="utf-8").lower()

print("\nКарта сохранена:", OUTPUT_PATH.resolve())
m